# Mix Produttivo Ottimo

## Dati
Abbiamo 
- Un insieme di **prodotti** $P = \{P1, P2, P3, P4, P5, P6\}$
- Un insieme di **materie prime** $M = \{\text{acciaio, legno, plastica, gomma, vetro, vernice}\}$
- Per ogni **prodotto** $p \in P$:
    - Un **profitto unitario** $PROF_p$
- Per ogni **materia prima** $m \in M$:
    - Una **disponibilità** massima $DISP_m$
- Per ogni coppia (materia prima, prodotto):
    - Un coefficiente di consumo $USO_{m,p}$ (quanta materia $m$ serve per un'unità di $p$)

## Obiettivo
Determinare le quantità $x_p$ da produrre per **massimizzare** il profitto totale rispettando i limiti di magazzino.

### Modello Matematico (ILP)

## Variabili di decisione
$x_p$: quantità (numero di pezzi) da produrre per il prodotto $p \in P$
- $x_p \in \mathbb{Z}^+ \quad (\text{ovvero } x_p \ge 0 \text{ e intero})$

## Parametri
- $PROF_p$: profitto unitario derivante dalla vendita del prodotto $p \in P$
- $DISP_m$: quantità massima disponibile per la materia prima $m \in M$
- $USO_{m,p}$: coefficiente tecnologico che indica il consumo della materia $m \in M$ per la produzione di un'unità del prodotto $p \in P$

## Funzione Obiettivo
$$\max Z = \sum_{p \in P} PROF_p \cdot x_p$$

## Soggetto a:
1. **Vincoli di disponibilità risorse (materie prime):**
$$\sum_{p \in P} USO_{m,p} \cdot x_p \le DISP_m \quad \forall m \in M$$

2. **Vincoli di integrità e non negatività:**
$$x_p \in \mathbb{Z}^+ \quad \forall p \in P$$
---

## Tabelle dei Dati

### 1. Profitti Unitari ($PROF_p$)
| P1 | P2 | P3 | P4 | P5 | P6 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 30 | 45 | 24 | 26 | 24 | 30 |

### 2. Matrice dei Consumi ($USO_{m,p}$) e Disponibilità ($DISP_m$)
| Materia Prima | P1 | P2 | P3 | P4 | P5 | P6 | **DISP** |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Acciaio** | 1.0 | 4.0 | 0.0 | 4.0 | 2.0 | 0.0 | **800** |
| **Legno** | 4.0 | 5.0 | 3.0 | 0.0 | 1.0 | 0.0 | **1160** |
| **Plastica**| 0.0 | 3.0 | 8.0 | 0.0 | 1.0 | 0.0 | **1780** |
| **Gomma** | 2.0 | 0.0 | 1.0 | 2.0 | 1.0 | 5.0 | **1050** |
| **Vetro** | 2.0 | 4.0 | 2.0 | 2.0 | 2.0 | 4.0 | **1360** |
| **Vernice** | 1.0 | 4.0 | 1.0 | 4.0 | 3.0 | 4.0 | **1240** |

In [1]:
import pyomo.environ as pyo

prodotti = ['P1','P2','P3','P4','P5','P6']
profitti = [30  , 45 , 24 , 26 , 24 , 30 ]
p = dict(zip(prodotti, profitti))

materie_prime= ['Acciaio', 'Legno', 'Plastica','Gomma','Vetro','Vernice']
disp=dict(zip(materie_prime,[800, 1160, 1780, 1050, 1360, 1240]))
mix_acciaio     = [1, 4, 0, 4, 2, 0]
mix_legno       = [4, 5, 3, 0, 1, 0]
mix_plastica    = [0, 3, 8, 0, 1, 0]
mix_gomma       = [2, 0, 1, 2, 1, 5]
mix_vetro       = [2, 4, 2, 2, 2, 4]
mix_vernice     = [1, 4, 1, 4, 3, 4]
mix_uso = dict(zip(materie_prime,[mix_acciaio, mix_legno, mix_plastica, mix_gomma, mix_vetro, mix_vernice]))

model= pyo.ConcreteModel(name="Mix Produttivo Ottimale")

model.x = pyo.Var(prodotti, within=pyo.NonNegativeIntegers)

def funzione_obiettivo(m:pyo.ConcreteModel) -> pyo.Expression:
    return pyo.quicksum(p[j]*m.x[j] for j in prodotti)
model.obiettivo= pyo.Objective(rule=funzione_obiettivo, sense=pyo.maximize)

#OCCHIO QUESTA NON MOLTO CHIARA
def vincoli_modello(m: pyo.ConcreteModel, materia: str) -> pyo.Expression:
    """
    Calcola la condizione 
    """
    somma_consumi = 0.0
    for j in range(len(prodotti)):
        nome_prodotto = prodotti[j]               
        quantita_necessaria = mix_uso[materia][j] 
        somma_consumi += quantita_necessaria * m.x[nome_prodotto]
    print(somma_consumi)
    return somma_consumi <= disp[materia]

model.vincoli = pyo.Constraint(materie_prime, rule=vincoli_modello)

ottimizzatore=pyo.SolverFactory('glpk').solve(model, tee=False)

model.display()

x[P1] + 4*x[P2] + 0*x[P3] + 4*x[P4] + 2*x[P5] + 0*x[P6]
4*x[P1] + 5*x[P2] + 3*x[P3] + 0*x[P4] + x[P5] + 0*x[P6]
0*x[P1] + 3*x[P2] + 8*x[P3] + 0*x[P4] + x[P5] + 0*x[P6]
2*x[P1] + 0*x[P2] + x[P3] + 2*x[P4] + x[P5] + 5*x[P6]
2*x[P1] + 4*x[P2] + 2*x[P3] + 2*x[P4] + 2*x[P5] + 4*x[P6]
x[P1] + 4*x[P2] + x[P3] + 4*x[P4] + 3*x[P5] + 4*x[P6]
Model Mix Produttivo Ottimale

  Variables:
    x : Size=6, Index={P1, P2, P3, P4, P5, P6}
        Key : Lower : Value : Upper : Fixed : Stale : Domain
         P1 :     0 : 120.0 :  None : False : False : NonNegativeIntegers
         P2 :     0 :   0.0 :  None : False : False : NonNegativeIntegers
         P3 :     0 : 220.0 :  None : False : False : NonNegativeIntegers
         P4 :     0 : 160.0 :  None : False : False : NonNegativeIntegers
         P5 :     0 :  20.0 :  None : False : False : NonNegativeIntegers
         P6 :     0 :  50.0 :  None : False : False : NonNegativeIntegers

  Objectives:
    obiettivo : Size=1, Index=None, Active=True
       